<a href="https://colab.research.google.com/github/zhengyangsu/Natural-Language-Processing/blob/main/latin_translation_lora.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Latin to English Translation with LoRA

Adapted from the Week 1 tutorial `movie_review_sentiment_classifier_with_bert.ipynb`  
Use a GPU runtime

# Setup Python Libraries (pip)

In [1]:
# transformers = models and tokenizers, peft = LoRA, bitsandbytes = 4-bit loading,
# accelerate = GPU placement, datasets = data loading, --quiet hides the long install log
!pip install transformers peft bitsandbytes accelerate datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 83.5 MB/s  0:00:00


# Mount Google Drive

In [2]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [3]:
import os

# Path to your NLP_Project folder in Google Drive
project_path = '/content/drive/MyDrive/NLP_Project'

# Create the folder if it does not exist
if not os.path.exists(project_path):
    os.makedirs(project_path)
    print(f"Created directory: {project_path}")

# Change the current working directory
os.chdir(project_path)
print(f"Current working directory set to: {os.getcwd()}")

Current working directory set to: /content/drive/MyDrive/NLP_Project


#Hugging Face Auth

In [ ]:
from huggingface_hub import notebook_login
from huggingface_hub import whoami

#notebook_login()
!hf auth login
!hf auth whoami
#print(whoami()['name'])

# Load the Dataset

In [4]:
from datasets import load_dataset  #downloads datasets

raw = load_dataset('grosenthal/latin_english_parallel')  #Latin-English pairs (train / test / valid splits) https://huggingface.co/datasets/grosenthal/latin_english_parallel

dataset = {  # a plain dict holding the three small subsets we will use
    'train': raw['train'].shuffle(seed=42).select(range(2000)),  # shuffle with a fixed seed, 2000 rows for training
    'val':   raw['valid'].select(range(200)),   # first 200 validation rows
    'test':  raw['test'].select(range(200)),    # first 200 test rows
}

#import json
'''
Final test data JSONL format
pairs = [
    {"la": "Arma virumque cano.", "en": "I sing of arms and the man."},
    {"la": "Carpe diem.", "en": "Seize the day."},
]'''

README.md:   0%|          | 0.00/1.87k [00:00<?, ?B/s]

data/train-00000-of-00001-9b65fddb561aaf(…): reconstructing file:   0%|          |  0.00B / 25.1MB            

data/train-00000-of-00001-9b65fddb561aaf(…): downloading bytes:           |  0.00B            

data/test-00000-of-00001-da9124f738a58df(…): reconstructing file:   0%|          |  0.00B /  261kB            

data/test-00000-of-00001-da9124f738a58df(…): downloading bytes:           |  0.00B            

data/valid-00000-of-00001-cf90f0473c819b(…): reconstructing file:   0%|          |  0.00B /  256kB            

data/valid-00000-of-00001-cf90f0473c819b(…): downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/99343 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1014 [00:00<?, ? examples/s]

Generating valid split:   0%|          | 0/1014 [00:00<?, ? examples/s]

'\nFinal test data JSONL format\npairs = [\n    {"la": "Arma virumque cano.", "en": "I sing of arms and the man."},\n    {"la": "Carpe diem.", "en": "Seize the day."},\n]'

# Tokenize the Dataset

In [5]:
from transformers import AutoTokenizer

checkpoint = 'hathibelagal/llama-3.2-latin'  #base model
tokenizer = AutoTokenizer.from_pretrained(checkpoint)  #tokenizer that matches the model's vocabulary
tokenizer.pad_token = tokenizer.eos_token  # Llama has no padding token, so reuse the end-of-text token

def tokenization(batch):  # function applied to a batch of rows
    # join each Latin sentence with its English translation into one prompt-style text
    texts = [f"Latin: {la}\nEnglish: {en}" for la, en in zip(batch['la'], batch['en'])]
    return tokenizer(texts, truncation=True, max_length=256)  # text -> token ids; cap to 256 tokens

tokenized = {k: v.map(tokenization, batched=True, remove_columns=v.column_names)  # run it on each split in batches, drop raw text columns
             for k, v in dataset.items()}  # loop over train / val / test

config.json:   0%|          | 0.00/838 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/50.6k [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.2MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/335 [00:00<?, ?B/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

Map:   0%|          | 0/200 [00:00<?, ? examples/s]

Map:   0%|          | 0/200 [00:00<?, ? examples/s]

# Load the Model and Add LoRA

In [6]:
import torch
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

bnb = BitsAndBytesConfig(load_in_4bit=True,  # store the frozen base weights in 4 bits
                         bnb_4bit_quant_type='nf4',  # NF4 is the 4-bit format designed for QLoRA
                         bnb_4bit_compute_dtype=torch.float16)  # compute in 16-bit floats

model = AutoModelForCausalLM.from_pretrained(checkpoint,  # download base model
                                             quantization_config=bnb,  # apply the 4-bit settings above
                                             device_map='auto')  # training on the GPU automatically

model = prepare_model_for_kbit_training(
    model,
    gradient_checkpointing_kwargs={'use_reentrant': False})  # casts some layers to full precision, cant update 4bit weight

model = get_peft_model(model, LoraConfig(r=16,  # rank of the LoRA matrices: higher = more capacity
                                         lora_alpha=32,  # scaling factor for the LoRA update
                                         target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj'],  # add LoRA to the attention layers
                                         task_type='CAUSAL_LM'))  # word-prediction with Parameter Efficient Fine Tuning（
model.print_trainable_parameters()  # show how few parameters are actually trained

model.safetensors.index.json:   0%|          | 0.00/20.9k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/180 [00:00<?, ?B/s]

trainable params: 9,175,040 || all params: 3,221,924,864 || trainable%: 0.2848


# Setup Training Configurations

In [7]:
from transformers import Trainer, TrainingArguments, DataCollatorForLanguageModeling  # training tools

training_args = TrainingArguments(
    seed=42,  # fixed random seed
    output_dir='./results',  # folder for chkpts
    num_train_epochs=1,  # total passes
    per_device_train_batch_size=4,  # 4 examples per step (small to fit GPU memory)
    gradient_accumulation_steps=4,  # accumulate 4 steps before each update = effective batch size 16
    learning_rate=2e-4,  # step size
    fp16=True,  # 16-bit training (T4 supports fp16 not bf16)
    eval_strategy='epoch',  # evaluate on the validation set at the end of each epoch
    logging_strategy="steps",
    logging_steps=10,
    report_to='none',  # do not log
)

trainer = Trainer(
    model=model,  # the LoRA model to train
    args=training_args,  # hyperparameter settings above
    train_dataset=tokenized['train'],  # data to learn from
    eval_dataset=tokenized['val'],  # data used for evaluation during training
    data_collator=DataCollatorForLanguageModeling(tokenizer, mlm=False),  # builds padded batches and labels; mlm=False = predict the next word
)

trainer.model.save_pretrained("./latin_lora_adapter")
tokenizer.save_pretrained("./latin_lora_adapter")

# Evaluate the Un-Fine-Tuned Model for a Baseline

In [8]:
import numpy as np  # for exp()

result = trainer.evaluate(tokenized['test'])  # loss on the test set (untrain LoRA starts at 0)
print(result, 'perplexity =', np.exp(result['eval_loss']))  # perplexity = exp(loss): lower = less random text

Training Loss,Validation Loss,Epoch
No log,3.219894,0


{'eval_loss': 3.2198944091796875} perplexity = 25.0254775811166


# Fine-Tune with LoRA

In [9]:
trainer.train()  # run the LoRA fine tuning loop, Traning loss loging step is 500

/usr/local/lib/python3.13/dist-packages/torch/_dynamo/eval_frame.py:1548: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Epoch,Training Loss,Validation Loss
1,No log,2.793176


/usr/local/lib/python3.13/dist-packages/torch/_dynamo/eval_frame.py:1548: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/python3.13/dist-packages/torch/_dynamo/eval_frame.py:1548: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/pyt

TrainOutput(global_step=125, training_loss=2.798360107421875, metrics={'train_runtime': 589.5015, 'train_samples_per_second': 3.393, 'train_steps_per_second': 0.212, 'total_flos': 4999518002601984.0, 'train_loss': 2.798360107421875, 'epoch': 1.0})

# Evaluate After Fine-Tuning

In [10]:
result = trainer.evaluate(tokenized['test'])  # same test set as the baseline, so the numbers are comparable
print(result, 'perplexity =', np.exp(result['eval_loss']))  # compare with the baseline perplexity

Training Loss,Validation Loss,Epoch
No log,2.794720,1


{'eval_loss': 2.794720411300659} perplexity = 16.358054586538767


# Try out some examples and save the adapter

In [11]:
def translate(la):  # translate one Latin sentence
    inputs = tokenizer(f"Latin: {la}\nEnglish:", return_tensors='pt').to(model.device)  # build prompt, make tensors, move to GPU
    out = model.generate(**inputs, max_new_tokens=100, do_sample=False)  # generate up to 100 new tokens, greedy (no randomness)
    text = tokenizer.decode(out[0], skip_special_tokens=True)  # token ids back to text
    return text.split('English:')[-1].split('\n')[0].strip()  # keep only the first line after "English:" and trim spaces

print(translate("Quo usque tandem abutere, Catilina, patientia nostra?"))  # try a famous Cicero line

model.save_pretrained('latin_lora')  # save only the small LoRA adapter, not the 3B base model

[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


How long will you continue to abuse our patience, Catiline?
